In [1]:
import folium
import pandas as pd
import geopandas as gpd
from glob import glob
from os.path import join

In [2]:
data_path = "/glade/derecho/scratch/ggantos/cat-rain-output/daily_sample/"
homeloc_zips_path = "/glade/derecho/scratch/cbecker/cat_testing_20250228/veraset_2024-03-15_2024-04-30.parquet"
path_zips = "/glade/work/cbecker/california_zip.geojson"

In [3]:
daily_files = sorted(glob(join(data_path, "veraset*")))
gdf_zips = gpd.read_file(path_zips)
homeloc_zips = pd.read_parquet(homeloc_zips_path, engine='pyarrow')
homeloc_zips.set_index('uid', inplace=True)
homeloc_zips.drop(['lat', 'lng'], axis=1, inplace=True)

In [ ]:
data_0 = pd.read_parquet(join(data_path, daily_files[0]), engine='pyarrow')
data_0.set_index('uid', inplace=True)
data_0 = data_0.assign(time=daily_files[0].split("veraset_")[1].split("_PST")[0])
data_0 = data_0.join(homeloc_zips, how='left')
data_0 = pd.merge(data_0.groupby("zipcode").sum(), gdf_zips, how='left', left_on="zipcode", right_on="ZIP_CODE")
data_0

In [4]:
data = []
for file in daily_files:
    data_f = pd.read_parquet(join(data_path, file), engine='pyarrow')
    data_f.set_index('uid', inplace=True)
    data_f = data_f.assign(time=file.split("veraset_")[1].split("_PST")[0])
    data_f = data_f.join(homeloc_zips, how='left')
    data_f = pd.merge(data_f.groupby("zipcode").sum(), gdf_zips, how='left', left_on="zipcode", right_on="ZIP_CODE")
    data.append(data_f)
data = pd.concat(data)

In [ ]:
lat_mean = gdf_subset['lat'].mean()
lng_mean = gdf_subset['lng'].mean()

In [ ]:
gdf_subset = gdf_subset.assign(time=gdf_subset["datetime"].dt.strftime('%Y-%m-%dT%H:%M:%SZ'))
gdf_subset = gdf_subset.assign(icon="circle")
gdf_subset = gdf_subset.assign(iconstyle={"color": "green",
                                          "weight": 1,
                                          "fillColor": "green",
                                          "fillOpacity": 0.6,
                                          "radius": 13})

In [ ]:
gdf_subset = gdf_subset[['time', 'horizontal_accuracy', 'geometry', 'icon', 'iconstyle']].to_json(drop_id=True)

duration: ISO 8601 duration format (https://www.digi.com/resources/documentation/digidocs/90001488-13/reference/r_iso_8601_duration_format.htm)

In [ ]:
m = folium.Map(location=[lat_mean, lng_mean],
               zoom_start=6)

folium.plugins.TimestampedGeoJson(gdf_subset,
                                  date_options="YYYY-MM-DD HH:MM",
                                  duration="P1D",
                                  transition_time=100,
                                  ).add_to(m)
m

![title](img/dots.png)